# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 3.1.0


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string
activity_year    string
lei              string
county_code      string
action_taken     string
loan_amount      string
income           string
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [3]:
def clean_export(source):
    out = source.copy(deep=True)

    # Проверяем разные версии одного номера
    known_key = out["source_row"].notna()
    versions = out.loc[known_key].drop_duplicates()
    assert versions["source_row"].is_unique, "Конфликт версий записи"

    # Исключаем одинаковые повторные доставки
    replay_mask = known_key & out["source_row"].duplicated()
    replay_n = int(replay_mask.sum())
    out = out.loc[~replay_mask].copy()

    # Убираем внешние пробелы территории
    out["county_code"] = out["county_code"].str.strip()

    # Преобразуем числа и считаем новые ошибки
    parse_errors = {}

    for field in ["source_row", "loan_amount", "income", "action_taken"]:
        numbers = pd.to_numeric(out[field], errors="coerce")
        failed = out[field].notna() & numbers.isna()
        parse_errors[field] = int(failed.sum())
        out[field] = numbers

    # Записываем причины карантина
    reason = pd.Series("", index=out.index, dtype="string")

    reason.loc[out["source_row"].isna()] += "unknown_source_row;"

    bad_amount = out["loan_amount"].isna() | out["loan_amount"].le(0)
    reason.loc[bad_amount] += "invalid_loan_amount;"

    bad_outcome = ~out["action_taken"].isin([1, 2, 3])
    reason.loc[bad_outcome] += "invalid_action_taken;"

    bad_county = ~out["county_code"].eq("11001").fillna(False)
    reason.loc[bad_county] += "invalid_county;"

    # Разделяем записи
    invalid = reason.ne("")
    clean = out.loc[~invalid].copy()

    quarantine = source.loc[out.index[invalid]].copy()
    quarantine["reason"] = reason.loc[invalid]

    # Собираем счетчики и проверяем баланс
    audit = {
        "input": len(source),
        "replays": replay_n,
        "clean": len(clean),
        "quarantine": len(quarantine),
        "parse_errors": parse_errors,
    }

    assert audit["input"] == (
        audit["clean"] + audit["quarantine"] + audit["replays"]
    )

    return clean, quarantine, audit

## H2. Год и известный доход

In [4]:
clean, quarantine, audit = clean_export(export)
assert clean["activity_year"].eq("2023").fillna(False).all()

income_known = clean["income"].notna().sum()
income_missing = clean["income"].isna().sum()
mean_income = clean["income"].mean()

print("Всего записей:", len(clean))
print("Доход известен:", income_known)
print("Доход неизвестен:", income_missing)
print(f"Средний известный доход: {mean_income:.2f} тыс. USD в год")

assert income_known + income_missing == len(clean)
assert audit["input"] == (
    audit["clean"] + audit["quarantine"] + audit["replays"]
)
assert_frame_equal(export, raw_copy)

print("Все проверки прошли")

Всего записей: 6828
Доход известен: 6485
Доход неизвестен: 343
Средний известный доход: 237.07 тыс. USD в год
Все проверки прошли


## H3. Проверка отчётного года

In [5]:
wrong_year = export.copy(deep=True)
year_column = wrong_year.columns.get_loc("activity_year")
wrong_year.iloc[0, year_column] = "2022"

year_ok = (
    wrong_year["activity_year"]
    .eq("2023")
    .fillna(False)
    .all()
)

print("Все годы правильные:", year_ok)

assert not year_ok
assert_frame_equal(export, raw_copy)

print("Неверный год обнаружен, исходная таблица не изменилась")

Все годы правильные: False
Неверный год обнаружен, исходная таблица не изменилась


## H+1. Конфликт исходных значений

In [6]:
conflict_export = export.copy(deep=True)

replay_mask = (
    conflict_export["source_row"].notna()
    & conflict_export["source_row"].duplicated()
)

last_replay_index = conflict_export.index[replay_mask][-1]

conflict_export.loc[last_replay_index, "loan_amount"] = "999999"

try:
    clean_export(conflict_export)
except AssertionError as error:
    print("Конфликт обнаружен:", error)
else:
    raise AssertionError("Ошибка проверки: конфликт не обнаружен")

assert_frame_equal(export, raw_copy)

Конфликт обнаружен: Конфликт версий записи


## H+2. Совпадение опубликованных полей

In [7]:
compare_columns = decisions.columns.drop("source_row")

without_matches = decisions.drop_duplicates(
    subset=compare_columns
)

rows_before = len(decisions)
rows_after = len(without_matches)

amount_before = decisions["loan_amount"].sum()
amount_after = without_matches["loan_amount"].sum()

print("Строк до:", rows_before)
print("Строк после:", rows_after)
print("Удалено строк:", rows_before - rows_after)

print()

print("Сумма до, USD:", amount_before)
print("Сумма после, USD:", amount_after)
print("Разница, USD:", amount_before - amount_after)

Строк до: 6830
Строк после: 6757
Удалено строк: 73

Сумма до, USD: 4577460000.0
Сумма после, USD: 4547855000.0
Разница, USD: 29605000.0
